<a href="https://colab.research.google.com/github/piehyun/crawling/blob/main/kakaomap_crawling_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 카카오맵 도로명 주소 및 지번 크롤링
## 코랩에서 드라이버 오류 나는 문제 해결

## 패키지 및 크롬 웹 드라이버 다운로드

In [ ]:
!python.exe -m pip install --upgrade pip
!pip install webdriver-manager selenium

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

!apt update -qq

!apt install -y -qq ./google-chrome-stable_current_amd64.deb

In [ ]:
!which google-chrome
!google-chrome --version

## 주소 리스트로 크롤링

In [5]:
# Google Colab + Selenium + KakaoMap 안정화 버전

import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Chrome 옵션 설정

options = Options()

options.binary_location = "/usr/bin/google-chrome"

# 최신 headless
options.add_argument("--headless=new")

# Colab 필수
options.add_argument("--no-sandbox")
options.add_argument("--disable-setuid-sandbox")
options.add_argument("--disable-dev-shm-usage")


options.add_argument("--disable-gpu")
options.add_argument("--disable-webgl")
options.add_argument("--disable-3d-apis")
options.add_argument("--disable-software-rasterizer")


options.add_argument("--disable-extensions")
options.add_argument("--disable-background-networking")
options.add_argument("--disable-sync")
options.add_argument("--no-first-run")
options.add_argument("--disable-default-apps")


options.add_argument("--remote-debugging-port=9222")


options.add_argument("--window-size=1920,1080")

options.add_argument("--disable-features=VizDisplayCompositor")



# Driver 실행
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)


# KakaoMap 접속
driver.get("https://map.kakao.com/")
time.sleep(5)

print("TITLE :", driver.title)
print("URL :", driver.current_url)



def clean_text(el):
    return (el.get_attribute("title") or el.text or "").strip()



# 첫 번째 검색 결과 주소 추출
def extract_first_address(driver, wait):

    first_item = wait.until(
        EC.visibility_of_element_located(
            (
                By.XPATH,
                "//*[@id='info.search.place.list']//li[contains(@class, 'PlaceItem')][1]"
            )
        )
    )

    # 주소 요소 대기
    wait.until(
        EC.visibility_of_element_located(
            (By.CSS_SELECTOR, "p[data-id='address']")
        )
    )

    address_el = first_item.find_element(
        By.CSS_SELECTOR,
        "p[data-id='address']"
    )

    lot_el = first_item.find_element(
        By.CSS_SELECTOR,
        "p[data-id='otherAddr']"
    )

    re_address = clean_text(address_el)
    re_num = clean_text(lot_el)

    # "(지번)" 제거
    re_num = re.sub(r"^\(지번\)\s*", "", re_num).strip()

    return re_address, re_num


# 테스트 데이터

addresses = [
    "아지트",
    "집",
    "당근",
    "토끼굴",
    "바나나",
    "오리농장",
    "커프",
    "비나레스토랑",
    "성심당"
]

re_df = pd.DataFrame({
    "address_keyword": addresses
})

re_df["re_address"] = ""
re_df["re_num"] = ""


# 서울 + 테스트 데이터, 기타 상황에 맞게 지역명 수정 가능

for i, keyword in re_df["address_keyword"].fillna("").items():

    keyword = f"서울 {keyword}".strip()

    try:

        print(f"\n검색중 : {keyword}")

        # 검색창 로딩 대기
        search_box = wait.until(
            EC.visibility_of_element_located(
                (By.ID, "search.keyword.query")
            )
        )

        # clear() 대신 사용
        search_box.send_keys(Keys.CONTROL + "a")
        search_box.send_keys(Keys.DELETE)

        time.sleep(0.5)

        # 검색어 입력
        search_box.send_keys(keyword)

        time.sleep(0.5)

        # 엔터
        search_box.send_keys(Keys.ENTER)

        # 카카오맵 로딩 대기
        time.sleep(3)

        # 주소 추출
        re_address, re_num = extract_first_address(driver, wait)

        re_df.at[i, "re_address"] = re_address
        re_df.at[i, "re_num"] = re_num

        print("도로명 :", re_address)
        print("지번 :", re_num)

    except Exception as e:

        print(f"[실패] {keyword}")
        print(e)

    # 요청 간격
    time.sleep(2)


# 결과 데이터프레임 head 출력

print("\n====================")
print(re_df.head(10))

# 종료
driver.quit()

TITLE : 카카오맵
URL : https://map.kakao.com/

검색중 : 서울 아지트
도로명 : 서울 강남구 강남대로 364 1층
지번 : 역삼동 826-21

검색중 : 서울 집
도로명 : 서울 중구 남대문로9길 12 나전빌딩 1층
지번 : 다동 117

검색중 : 서울 당근
도로명 : 서울 서초구 강남대로 465 교보타워 7,10-12층
지번 : 서초동 1303-22

검색중 : 서울 토끼굴
도로명 : 서울 강남구 압구정동 495
지번 : 

검색중 : 서울 바나나
도로명 : 서울 성동구 아차산로7길 15-8
지번 : 성수동2가 289-75

검색중 : 서울 오리농장
도로명 : 서울 강북구 오패산로 113
지번 : 미아동 49-101

검색중 : 서울 커프
도로명 : 서울 강남구 테헤란로38길 40 1층
지번 : 역삼동 722-2

검색중 : 서울 비나레스토랑
도로명 : 서울 동대문구 안암로24길 4
지번 : 제기동 137-41

검색중 : 서울 성심당
도로명 : 서울 중구 남대문로 30
지번 : 남창동 1-2

  address_keyword                     re_address        re_num
0             아지트             서울 강남구 강남대로 364 1층    역삼동 826-21
1               집        서울 중구 남대문로9길 12 나전빌딩 1층        다동 117
2              당근  서울 서초구 강남대로 465 교보타워 7,10-12층   서초동 1303-22
3             토끼굴                서울 강남구 압구정동 495              
4             바나나             서울 성동구 아차산로7길 15-8  성수동2가 289-75
5            오리농장                서울 강북구 오패산로 113    미아동 49-101
6              커프           서울 